# Vector Store

Notebook 2 computed cosine similarity by hand over every paper — fine for a
demo, not something to repeat by hand at scale. This notebook wraps the same
dot product behind `NumpyStore`, one implementation of a `VectorStore`
protocol, so notebook 5's BM25 index can sit behind the exact same shape.

**The chunks, again.** Reuse notebook 1's `small_to_big` strategy and
notebook 2's on-disk cache — nothing in this cell should reach the API.

In [1]:
from readnext.config import PAPERS_FILE
from readnext.corpus import chunk, load
from readnext.embed import embed_texts

papers = load(PAPERS_FILE)
chunks = chunk(papers, strategy="small_to_big")
vectors = embed_texts([c.text for c in chunks])
len(papers), len(chunks), vectors.shape

(1693, 12320, (12320, 1536))

**Build the index.** `build_dense_index` calls `NumpyStore.add` once, with
one row of metadata per chunk. From here on, everything — search, filters,
ablations — talks to the store, never to `chunks` or `vectors` directly.

In [2]:
from readnext.search import build_dense_index

store = build_dense_index(chunks, vectors)
len(store)

12320

**`search()` end to end.** Embed the query, retrieve the top
`candidates_k` chunks, collapse them to one hit per paper by best chunk
score, and cut to `k`. This is the whole dense pipeline — `recommend()` will
later wrap it with filters, reranking, and diversity, but the retrieval
underneath stays exactly this.

In [3]:
from readnext.search import search

by_id = {p.id: p for p in papers}

def show(query: str, k: int = 5) -> None:
    for hit in search(store, query, k=k):
        print(f"{hit.score:.3f}  {by_id[hit.paper_id].title}")

show("retrieval augmented generation for question answering")

0.754  What Makes a Good Fiqh Retriever? Answer Retrieval for Arabic Islamic Jurisprudence
0.695  Bridging the Question-Answer Gap in Retrieval-Augmented Generation: Hypothetical Prompt Embeddings
0.693  Beyond Self-Knowledge: Propagating Uncertainty Across Reasoning and Retrieval in LLMs
0.686  Cost Scales with Change, Not Corpus Size: Incrementally Maintaining an Evolving Semantic Substrate
0.680  VDGR-RAG: Vectors, Directories, Graphs, and Reflection Are All You Need for Unified Reasoning over Hierarchical Enterprise Knowledge


**A second query, by eye.** Different topic, same store — a sanity check
that the first result wasn't a fluke of one query.

In [4]:
show("reinforcement learning from human feedback")

0.578  Learning Generalizable Behaviors for Terminal Agents
0.576  Beyond Imitation: Self-Improving Robot Policies via Off-Policy Q-Planning
0.562  The Chase Is the Curriculum, the Capture Anchors the Credit: Pursuit-Evasion Self-Play for Zero-Data LLM Reasoning
0.544  Structure-aware Relative Policy Optimization for Ranking
0.536  Rethinking Demonstration Unlearning in Imitation Learning for Robotics


**Dimension reduction.** `text-embedding-3-small` takes a `dimensions`
parameter — the endpoint truncates its native embedding before returning it,
no separate PCA step needed. Real recall/nDCG needs a labeled set, which
doesn't exist until notebook 4's golden set, so this ablation measures what's
already measurable: storage, search speed, and how much the ranking itself
shifts. This cell makes one real (small — a few cents at most) paid call over
the whole corpus, distinct from the cache notebook 2 already paid for.

In [5]:
reduced_vectors = embed_texts([c.text for c in chunks], dimensions=512)
reduced_vectors.shape

(12320, 512)

**Space.** Half the dimensions is exactly half the bytes. At ~2,000 papers
that's nothing, but it's the number that matters once the corpus is large
enough to need a real vector database.

In [6]:
full_bytes, reduced_bytes = vectors.nbytes, reduced_vectors.nbytes
full_bytes, reduced_bytes, f"{reduced_bytes / full_bytes:.0%} of full size"

(75694080, 25231360, '33% of full size')

**Speed.** Build a second store at the reduced dimension, then time the
same brute-force search over each — fewer dimensions means fewer
multiplications per row of the dot product.

In [7]:
import time

reduced_store = build_dense_index(chunks, reduced_vectors)

def time_search(store, query_vector, k=40, repeats=20):
    t0 = time.time()
    for _ in range(repeats):
        store.search(query_vector, k=k)
    return (time.time() - t0) / repeats * 1000  # ms

query_text = "retrieval augmented generation for question answering"
full_query_vector = embed_texts([query_text])[0]
reduced_query_vector = embed_texts([query_text], dimensions=512)[0]

full_ms = time_search(store, full_query_vector)
reduced_ms = time_search(reduced_store, reduced_query_vector)
f"{full_ms:.2f}ms at 1536 dims vs {reduced_ms:.2f}ms at 512 dims"

'6.18ms at 1536 dims vs 2.83ms at 512 dims'

**Does the ranking change?** No golden set yet to score against, so the
honest proxy is agreement: for the same query, how much does the reduced
index's top 10 overlap with the full index's?

In [8]:
def top_paper_ids(store, query_text, k=10, dimensions=None):
    return [hit.paper_id for hit in search(store, query_text, k=k, dimensions=dimensions)]

queries = [
    "retrieval augmented generation for question answering",
    "reinforcement learning from human feedback",
]
for q in queries:
    full_ids = top_paper_ids(store, q)
    reduced_ids = top_paper_ids(reduced_store, q, dimensions=512)
    overlap = len(set(full_ids) & set(reduced_ids))
    print(f"{overlap}/10 overlap — {q}")

9/10 overlap — retrieval augmented generation for question answering
9/10 overlap — reinforcement learning from human feedback


**Done.** `search(store, query_text, k=10)` returns sensible papers,
`VectorStore` is a protocol with `NumpyStore` as its only implementation, and
the dimension ablation has a real (if provisional) answer: at 512 dimensions
the index is half the size and faster to search, and the top-10 rankings
mostly agree with the full-dimension index on both test queries. Whether that
agreement is *good enough* — i.e. whether it costs recall against real
relevance judgments — isn't answerable until notebook 4's golden set exists to
check it against. Notebook 5 adds a BM25 index behind the same
`VectorStore`-shaped interface.